In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorowanie lub anulowanie zadania

Wyświetl listę swoich zadań na [stronie Obciążenia](https://quantum.cloud.ibm.com/workloads).

## Sprawdź status zadania
Przejdź do [tabeli Obciążenia](https://quantum.cloud.ibm.com/workloads) i sprawdź kolumnę Status, aby zobaczyć, czy zadanie zostało ukończone czy zakończyło się błędem.

## Sprawdź pozostałe wykorzystanie
Przejdź do [tabeli Instancje](https://quantum.cloud.ibm.com/instances) i wybierz kartę odpowiadającą planowi, dla którego chcesz sprawdzić pozostałe wykorzystanie. Wyświetlany jest całkowity czas wykorzystany oraz całkowity czas pozostały w ramach planu.

## Przeglądaj metryki liczby przesłanych zadań i obciążeń
Przejdź na [stronę Analityka](https://quantum.cloud.ibm.com/analytics), aby zobaczyć całkowitą liczbę przesłanych zadań, a także liczbę obciążeń wsadowych i obciążeń Session. Pamiętaj, że stronę Analityka możesz zobaczyć tylko dla kont, których jesteś właścicielem lub którymi zarządzasz.

## Monitoruj zadanie
Użyj instancji zadania, aby sprawdzić jego status lub pobrać wyniki, wywołując odpowiednie polecenie:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Przeglądaj wyniki zadania natychmiast po jego ukończeniu. Wyniki zadania są dostępne po jego zakończeniu. Dlatego `job.result()` jest wywołaniem blokującym aż do ukończenia zadania.                               |
| job.job\_id()                 | Zwraca identyfikator jednoznacznie identyfikujący dane zadanie. Pobranie wyników zadania w późniejszym czasie wymaga identyfikatora zadania. Zaleca się więc zapisywanie identyfikatorów zadań, które możesz chcieć pobrać później. |
| job.status()                  | Sprawdza status zadania.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Pobiera zadanie, które wcześniej przesłałeś(-aś). To wywołanie wymaga identyfikatora zadania.                                                                                                                                      |

<span id="retrieve-later"></span>
## Pobierz wyniki zadania w późniejszym czasie
Wywołaj `service.job(\<job\_id>)`, aby pobrać zadanie, które wcześniej przesłałeś(-aś). Jeśli nie masz identyfikatora zadania lub chcesz pobrać wiele zadań naraz — w tym zadania z wycofanych QPU (jednostek przetwarzania kwantowego) — wywołaj zamiast tego `service.jobs()` z opcjonalnymi filtrami. Zobacz [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` zwraca również zadania uruchomione za pomocą wycofanego pakietu `qiskit-ibm-provider`. Zadania przesłane przez starszy (również wycofany) pakiet `qiskit-ibmq-provider` nie są już dostępne.

### Przykład
Ten przykład zwraca 10 ostatnich zadań runtime uruchomionych na `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>